In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Importing Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# Device GPU Check

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Defining Paths

In [ ]:
train_path = 'chest_xray/train'
val_path = 'chest_xray/val'
test_path = 'chest_xray/test'

# Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# Loading Dataset

In [ ]:
train_data = datasets.ImageFolder(train_path, transform=train_transform)
val_data = datasets.ImageFolder(val_path, transform=val_transform)
test_data = datasets.ImageFolder(test_path, transform=val_transform)

print(train_data.classes)

['NORMAL', 'PNEUMONIA']


# DataLoader

In [ ]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=32, num_workers=0)
test_loader = DataLoader(test_data, batch_size=32, num_workers=0)

# Model (ResNet18) - Initial Training

In [ ]:
model = models.resnet18(pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer (ONLY HERE)
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


# Loss + Optimizer (Initial)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Training (Initial)

In [ ]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.2722
Epoch 2, Loss: 0.1749
Epoch 3, Loss: 0.1548
Epoch 4, Loss: 0.1408
Epoch 5, Loss: 0.1424


# Evaluation

In [ ]:
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

# Checking Basline Accuracy

In [ ]:
print("Validation Accuracy:", evaluate(val_loader))
print("Test Accuracy:", evaluate(test_loader))

Validation Accuracy: 75.0
Test Accuracy: 80.44871794871794


# Fine Tuning

In [ ]:
# Freeze all
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last block
for param in model.layer4.parameters():
    param.requires_grad = True

# Add dropout (important)
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 2)
)

model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Training (Fine Tuning)

In [ ]:
epochs = 3
best_val = 0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    val_acc = evaluate(val_loader)

    print(f"Epoch {epoch+1} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    # Saving best model
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), "best_model.pth")

Epoch 1 | Loss: 0.1257 | Train Acc: 95.36% | Val Acc: 87.50%
Epoch 2 | Loss: 0.0372 | Train Acc: 98.68% | Val Acc: 93.75%
Epoch 3 | Loss: 0.0171 | Train Acc: 99.58% | Val Acc: 93.75%


# Loading Best Model

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))

<All keys matched successfully>

# Final Evaluation

In [ ]:
print("Final Validation Accuracy:", evaluate(val_loader))
print("Final Test Accuracy:", evaluate(test_loader))

Final Validation Accuracy: 93.75
Final Test Accuracy: 83.01282051282051


# “The model achieved ~93% validation accuracy and ~83% test accuracy. The difference is due to the small validation set and slight overfitting, which I controlled using dropout and early stopping.”